# MCGS → geometric GS → conformal 2D mesh

Tracked as `confMesh2d_mcgs.ipynb`.

Teaching path from a Monte-Carlo labelled image to a grain-boundary-conformant FE mesh.

Uses **`confMesh2dGMSH`** (raw Gmsh). The pygmsh class `confMesh2d` is deprecated.

For a mesh-only walkthrough (no MCGS), see `confMesh2d_gmsh.ipynb`. Requires `gmsh` (`pip install upxo[mesh]`).

This uses **Technique A** (`polygonised_grain_structure`); see `confMesh2d_technique_b.ipynb` for the Voronoi-based alternative and `confMesh2d_compare_techniques.ipynb` for a direct comparison. A grain fully enclosed by another grain (an island) is handled automatically; how the resulting host-with-a-hole plus separate island polygon gets meshed is shown in isolation in `confMesh2d_islands_voids.ipynb`.

## STEP 1 — Grain structure generation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from upxo.ggrowth.mcgs import mcgs
from upxo.pxtal.geometrification import polygonised_grain_structure
from upxo.meshing.conformal_mesher2d import confMesh2dGMSH
from upxo.meshing.writer_ABQ import summarize_inp
from pathlib import Path
_XLS = next(
    (p / "src" / "upxo" / "demos" / "confMesh" / "confMesh1.xls"
     for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "src" / "upxo" / "demos" / "confMesh" / "confMesh1.xls").is_file()),
    Path("confMesh1.xls"),
)
print("dashboard:", _XLS)

### Point the solver suite to the input dashboard excel filke

In [ ]:
pxt = mcgs(input_dashboard=str(_XLS))

### Grow the grains

In [ ]:
pxt.simulate()

### Detect the grains

In [ ]:
pxt.detect_grains(library="cc3d", connectivity=4)

### Select a temporal slice

In [ ]:
pxt.gs

In [ ]:
pxt.gs[9]

In [ ]:
gstslice = pxt.gs[list(pxt.gs.keys())[-1]]

### Characteruise

In [ ]:
gstslice.char_morph_2d(
    bbox=True, use_version=2, bbox_ex=True, area=True, eq_diameter=True,
    perimeter=True, aspect_ratio=True, solidity=True, feret_diameter=True,
    saa=True, char_gb=True, make_skim_prop=True, throw=False, append=False)

# Extract the neighbourhood dataset of all grains in this temporal sloce.
gstslice.find_neigh()

In [ ]:
from upxo._sup.raster_islands import find_island_regions
_, _filled_labels, _ = find_island_regions(gstslice.lgi)
n_islands = len(np.unique(gstslice.lgi)) - len(_filled_labels)
print("grains", len(np.unique(gstslice.lgi)) - 1,
      "| island grains (fully enclosed by another grain):", n_islands)

In [ ]:
plt.imshow(gstslice.lgi)
plt.show()

## STEP 2 — Pixelated GS to geometric GS

`polygonised_grain_structure` parses `lgi` / grain IDs / neighbours into per-grain polygons and a GB network.

lfi - local feature id

In [ ]:
geomGS = polygonised_grain_structure(gstslice.lgi, gstslice.gid, gstslice.neigh_gid)

In [ ]:
geomGS.pix_to_geom(verbose=False, perform_xtal_dict_check=True)

In [ ]:
fig, ax = geomGS.plotgs(geomGS.POLYXTAL, figsize=(5, 5), dpi=110)
ax.set_title("raw polygons")

A grain drawn with a visible hole in this plot is a host: an island grain (counted above) is its own separate polygon exactly filling that hole, not part of the host's polygon.

## STEP 3 — Grain-boundary smoothing

Smoothing shortens stair-step pixel boundaries so Gmsh can place conformal GB edges.

In [ ]:
npasses = 2
min_segment_length_factor = 2
gsname = f"gs.{npasses}passes.{min_segment_length_factor}minseglenfactor"

In [ ]:
geomGS.smooth_gbsegs(
    geomGS.GB, npasses=npasses,
    max_smooth_levels=np.repeat(min_segment_length_factor, npasses),
    plot=False, name=gsname)

In [ ]:
pxtal = geomGS.smoothed[gsname]["POLYXTAL"]

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 5), dpi=200)
geomGS.plotgs(geomGS.POLYXTAL, fig=fig, ax=axes[0])
axes[0].set_title("raw")
axes[0].set_axis_off()
geomGS.plotgs(pxtal, fig=fig, ax=axes[1])
axes[1].set_title("smoothed")
axes[1].set_axis_off()

## STEP 4 — Conformal FE mesh (`confMesh2dGMSH`)

`from_geometric_pxtal` is the drop-in replacement for the old pygmsh constructor. `femesh_gmsh` talks to the raw Gmsh API.

In [ ]:
gsConfMesh = confMesh2dGMSH.from_geometric_pxtal(
    gsgen_method="shapely_pxtal_load",
    pxtal=pxtal,
    xbound=[gstslice.uigrid.xmin, gstslice.uigrid.xmax],
    ybound=[gstslice.uigrid.ymin, gstslice.uigrid.ymax])

In [ ]:
gsConfMesh.femesh_gmsh(
    mesh_size_gb=0.5, mesh_size_bulk=1.0,
    mesh_algo=8, mesh_order=1, recombine_to_quads=False)

In [ ]:
gsConfMesh.form_elsets_gmsh()

In [ ]:
gsConfMesh.build_boundary_nsets()

In [ ]:
gsConfMesh.build_gb_nset()

In [ ]:
# gsConfMesh.validation_report

In [ ]:
vr = gsConfMesh.validation_report
print(f"elements: {vr['n_tri']} tri + {vr['n_quad']} quad | "
      f"grains missing elements: {len(vr['grains_missing_elements'])} (target 0) | "
      f"inverted elements: {vr['clockwise_elements']} (target 0) | "
      f"degenerate elements: {vr['degenerate_elements']} (target 0)")

Grain-coloured fill and face NSETs come from `plot_by_grain` (`upxo.viz.meshviz`). Abaqus export is `export_abaqus_inp` (`upxo.meshing.writer_ABQ`): CPS4/CPS3 for `plane='stress'`, CPE4/CPE3 for `'strain'`.

In [ ]:
fig, ax = gsConfMesh.plot_by_grain(
    figsize=(6, 6), show_gb=True, show_nsets=True, dpi=200,
    title="MCGS conformal mesh by grain")
# fig

In [ ]:
out = Path.cwd() / "confMesh2d_mcgs_out"
inp = gsConfMesh.export_abaqus_inp(out / "rve_cps.inp", plane="stress")
inp, summarize_inp(inp)

### UPXO Citation

Sunil Anandatheertha, Vikram Phalke, Eralp Demir, Chris Hardie, UKAEA Poly-XTAL operations (UPXO V1.0): An open-source Python package for generating, assessment and meshing of poly-crystalline grain structures, SoftwareX, Volume 34, 2026, 102736, ISSN 2352-7110, https://doi.org/10.1016/j.softx.2026.102736